In [1]:
import formulallm.formula as f

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent, FunctionCallingAgentWorker

In [7]:
from llama_index.llms.ollama import Ollama

llama3 = Ollama(model="llama3", base_url='http://localhost:11434', temperature=0.0, request_timeout=600)

True

In [10]:
def load(filePath: str) -> str:
    """Load the Formula DSL code from the file path"""
    f.load()

load_tool = FunctionTool.from_defaults(fn=load)

In [11]:
def solve():
    """Try to solve the partial model based on the domain constraints"""
    f.solve("pm","1","Mapping.conforms")

solve_tool = FunctionTool.from_defaults(fn=solve)

In [12]:
def list():
    """Get a list of all the tasks run so far"""
    f.list()

list_tool = FunctionTool.from_defaults(fn=list)

In [13]:
def extract(task_id: str):
    """Extract the solve result to the task with task_id"""
    """If the partial model is solvable, will return the solution"""
    """Else if the partial model is unsolvable, will return the least unsatisfied core conditions"""
    """The task_id is a string representing an integer which is incremented by 1 each time we run the solve command"""
    f.extract(task_id,"0","0")

extract_result_tool = FunctionTool.from_defaults(fn=extract)

In [12]:
class FormulaFixerAgent:
    def __init__(
        self,
        tools: Sequence[BaseTool] = [],
        llm: OpenAI = OpenAI(temperature=0, model="gpt-3.5-turbo-0613"),
        chat_history: List[ChatMessage] = [],
    ) -> None:
        self._llm = llm
        self._tools = {tool.metadata.name: tool for tool in tools}
        self._chat_history = chat_history

    def reset(self) -> None:
        self._chat_history = []

    def chat(self, message: str) -> str:
        chat_history = self._chat_history
        chat_history.append(ChatMessage(role="user", content=message))
        tools = [
            tool.metadata.to_openai_tool() for _, tool in self._tools.items()
        ]

        ai_message = self._llm.chat(chat_history, tools=tools).message
        additional_kwargs = ai_message.additional_kwargs
        chat_history.append(ai_message)

        tool_calls = additional_kwargs.get("tool_calls", None)
        # parallel function calling is now supported
        if tool_calls is not None:
            for tool_call in tool_calls:
                function_message = self._call_function(tool_call)
                chat_history.append(function_message)
                ai_message = self._llm.chat(chat_history).message
                chat_history.append(ai_message)

        return ai_message.content

    def _call_function(
        self, tool_call: ChatCompletionMessageToolCall
    ) -> ChatMessage:
        id_ = tool_call.id
        function_call = tool_call.function
        tool = self._tools[function_call.name]
        output = tool(**json.loads(function_call.arguments))
        return ChatMessage(
            name=function_call.name,
            content=str(output),
            role="tool",
            additional_kwargs={
                "tool_call_id": id_,
                "name": function_call.name,
            },
        )

In [13]:
agent = FormulaFixerAgent(tools=[load_tool, solve_tool])

In [14]:
agent.chat("Hi")

Retrying llama_index.llms.openai.base.OpenAI._chat in 0.15205690844761266 seconds as it raised APIConnectionError: Connection error..
Retrying llama_index.llms.openai.base.OpenAI._chat in 1.5696152237704466 seconds as it raised APIConnectionError: Connection error..


APIConnectionError: Connection error.